In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Brain_Tumor_Att_Dataset"  # adjust if BRISC is in a subfolder
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    if level > 3:
        continue
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:
        print(f'{subindent}{file}')

In [ ]:
import os

# Start from MyDrive and find the dataset
base = "/content/drive/MyDrive"
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    if level > 5:
        continue
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level == 5 and files:
        subindent = ' ' * 2 * (level + 1)
        for f in files[:3]:
            print(f'{subindent}{f}')

In [ ]:
import os

base = "/content/drive/MyDrive/Brain_Tumor_Att_Dataset/brisc2025"

seg_images = os.listdir(f"{base}/segmentation_task/train/images")
seg_masks = os.listdir(f"{base}/segmentation_task/train/masks")

print(f"Train images: {len(seg_images)}")
print(f"Train masks: {len(seg_masks)}")

# Check if IDs match
img_ids = set(f.replace('.jpg','') for f in seg_images)
mask_ids = set(f.replace('.png','') for f in seg_masks)

print(f"\nIDs in images but not masks: {len(img_ids - mask_ids)}")
print(f"IDs in masks but not images: {len(mask_ids - img_ids)}")

# Sample pair
print(f"\nSample image: {seg_images[0]}")
print(f"Sample mask:  {seg_masks[0]}")

In [ ]:
seg_images_sorted = sorted(seg_images)
seg_masks_sorted = sorted(seg_masks)

for i in range(3):
    img = seg_images_sorted[i].replace('.jpg', '')
    mask = seg_masks_sorted[i].replace('.png', '')
    match = "✓" if img == mask else "✗ MISMATCH"
    print(f"{match}  {seg_images_sorted[i]}  ←→  {seg_masks_sorted[i]}")

In [ ]:
import shutil

print("Copying dataset to local disk...")
shutil.copytree(
    "/content/drive/MyDrive/Brain_Tumor_Att_Dataset",
    "/content/Brain_Tumor_Att_Dataset"
)
print("Done!")

In [ ]:
# ── Installs ─────────────────────────────────────────────────────────
!pip install segmentation-models-pytorch albumentations -q

# ── Imports ──────────────────────────────────────────────────────────
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from collections import Counter

# ── Paths & Maps ─────────────────────────────────────────────────────
base = "/content/Brain_Tumor_Att_Dataset/brisc2025"

SEG_CLASS_MAP = {"gl": 0, "me": 1, "pi": 2}        # 3 classes, no no_tumor
CLS_CLASS_MAP = {"glioma": 0, "meningioma": 1, "no_tumor": 2, "pituitary": 3}

# ── Transforms ───────────────────────────────────────────────────────
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# ── Segmentation Dataset ──────────────────────────────────────────────
class SegmentationDataset(Dataset):
    def __init__(self, split="train", transform=None):
        self.transform = transform
        img_dir  = f"{base}/segmentation_task/{split}/images"
        mask_dir = f"{base}/segmentation_task/{split}/masks"

        self.pairs = []
        for fname in sorted(os.listdir(img_dir)):
            img_path  = os.path.join(img_dir, fname)
            mask_path = os.path.join(mask_dir, fname.replace('.jpg', '.png'))
            if os.path.exists(mask_path):
                cls_code = fname.split('_')[3]  # fixed: index 3
                label = SEG_CLASS_MAP.get(cls_code, 0)
                self.pairs.append((img_path, mask_path, label))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.pairs[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path).convert("L"))
        mask  = (mask > 127).astype(np.float32)

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask  = aug["mask"].unsqueeze(0)

        return image, mask, torch.tensor(label, dtype=torch.long)

# ── Classification Dataset ────────────────────────────────────────────
class ClassificationDataset(Dataset):
    def __init__(self, split="train", transform=None):
        self.transform = transform
        self.samples   = []
        cls_dir = f"{base}/classification_task/{split}"

        for class_name, label in CLS_CLASS_MAP.items():
            folder = os.path.join(cls_dir, class_name)
            for fname in sorted(os.listdir(folder)):
                if fname.endswith('.jpg'):
                    self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))

        if self.transform:
            aug   = self.transform(image=image)
            image = aug["image"]

        return image, torch.tensor(label, dtype=torch.long)

# ── Weighted Sampler for imbalanced classification ────────────────────
def make_weighted_sampler(dataset):
    labels = [s[1] for s in dataset.samples]
    counts = Counter(labels)
    weights = [1.0 / counts[l] for l in labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# ── Datasets ──────────────────────────────────────────────────────────
seg_train_dataset = SegmentationDataset("train", train_transform)
seg_test_dataset  = SegmentationDataset("test",  val_transform)
cls_train_dataset = ClassificationDataset("train", train_transform)
cls_test_dataset  = ClassificationDataset("test",  val_transform)

# ── DataLoaders ───────────────────────────────────────────────────────
seg_train_loader = DataLoader(seg_train_dataset, batch_size=8, shuffle=True,
                              num_workers=2, pin_memory=True)
seg_test_loader  = DataLoader(seg_test_dataset,  batch_size=8, shuffle=False,
                              num_workers=2, pin_memory=True)
cls_train_loader = DataLoader(cls_train_dataset, batch_size=8,
                              sampler=make_weighted_sampler(cls_train_dataset),
                              num_workers=2, pin_memory=True)
cls_test_loader  = DataLoader(cls_test_dataset,  batch_size=8, shuffle=False,
                              num_workers=2, pin_memory=True)

# ── Sanity check ──────────────────────────────────────────────────────
images, masks, labels = next(iter(seg_train_loader))
print(f"Seg image:  {images.shape}")
print(f"Seg mask:   {masks.shape}")
print(f"Seg labels: {labels}")

images, labels = next(iter(cls_train_loader))
print(f"Cls image:  {images.shape}")
print(f"Cls labels: {labels}")

In [ ]:
import segmentation_models_pytorch as smp
class MultiTaskUNet(nn.Module):
    def __init__(self, num_seg_classes=3, num_cls_classes=4):
        super().__init__()

        self.encoder = smp.encoders.get_encoder(
            "efficientnet-b3",
            in_channels=3,
            depth=5,
            weights="imagenet"
        )

        self.decoder = smp.decoders.unetplusplus.decoder.UnetPlusPlusDecoder(
            encoder_channels=self.encoder.out_channels,
            decoder_channels=(256, 128, 64, 32, 16),
            n_blocks=5,
            use_norm='batchnorm',        # fixed
            center=False,
            attention_type="scse"
        )

        self.seg_head = smp.base.SegmentationHead(
            in_channels=16,
            out_channels=num_seg_classes,
            activation=None,
            kernel_size=3
        )

        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(self.encoder.out_channels[-1], 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_cls_classes)
        )

    def forward(self, x):
        features    = self.encoder(x)
        decoder_out = self.decoder(features)
        seg_mask    = self.seg_head(decoder_out)
        cls_out     = self.cls_head(features[-1])
        return seg_mask, cls_out


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

model = MultiTaskUNet(num_seg_classes=3, num_cls_classes=4).to(device)

dice_loss = smp.losses.DiceLoss(mode='multiclass', classes=3)
ce_seg    = nn.CrossEntropyLoss()
ce_cls    = nn.CrossEntropyLoss()

def combined_loss(seg_pred, seg_true, cls_pred, cls_true, alpha=0.5, beta=0.5):
    seg_true_sq = seg_true.squeeze(1).long()
    seg_l = dice_loss(seg_pred, seg_true_sq) + ce_seg(seg_pred, seg_true_sq)
    cls_l = ce_cls(cls_pred, cls_true)
    return alpha * seg_l + beta * cls_l

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 256, 256).to(device)
    seg_out, cls_out = model(dummy)
    print(f"Seg output: {seg_out.shape}")
    print(f"Cls output: {cls_out.shape}")
    print("Model init successful ✓")

In [ ]:
!df -h /content

In [ ]:
from tqdm import tqdm

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for seg_imgs, masks, seg_labels in tqdm(loader, leave=False):
        seg_imgs   = seg_imgs.to(device)
        masks      = masks.to(device)
        seg_labels = seg_labels.to(device)

        seg_pred, cls_pred = model(seg_imgs)
        loss = combined_loss(seg_pred, masks, cls_pred, seg_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def val_epoch(model, loader):
    model.eval()
    total_loss  = 0
    correct     = 0
    total       = 0
    dice_scores = []

    with torch.no_grad():
        for seg_imgs, masks, seg_labels in loader:
            seg_imgs   = seg_imgs.to(device)
            masks      = masks.to(device)
            seg_labels = seg_labels.to(device)

            seg_pred, cls_pred = model(seg_imgs)
            loss = combined_loss(seg_pred, masks, cls_pred, seg_labels)
            total_loss += loss.item()

            # classification accuracy
            preds    = cls_pred.argmax(dim=1)
            correct += (preds == seg_labels).sum().item()
            total   += seg_labels.size(0)

            # mean dice across 3 classes
            seg_true_sq = masks.squeeze(1).long()
            seg_prob    = torch.softmax(seg_pred, dim=1)
            for c in range(3):
                pred_c = (seg_prob.argmax(dim=1) == c).float()
                true_c = (seg_true_sq == c).float()
                inter  = (pred_c * true_c).sum()
                dice   = (2 * inter) / (pred_c.sum() + true_c.sum() + 1e-8)
                dice_scores.append(dice.item())

    return (total_loss / len(loader),
            correct / total,
            np.mean(dice_scores))


# ── Training loop ─────────────────────────────────────────────────────
EPOCHS    = 30
best_dice = 0.0
history   = {"train_loss": [], "val_loss": [], "val_acc": [], "val_dice": []}

for epoch in range(EPOCHS):
    train_loss              = train_epoch(model, seg_train_loader, optimizer)
    val_loss, val_acc, val_dice = val_epoch(model, seg_test_loader)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_dice"].append(val_dice)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train: {train_loss:.4f} | "
          f"Val: {val_loss:.4f} | "
          f"Acc: {val_acc:.4f} | "
          f"Dice: {val_dice:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "/content/drive/MyDrive/best_model.pth")
        print(f"  → best model saved (dice: {best_dice:.4f})")


LOADING THE SAVED MODEL

In [ ]:
checkpoint = torch.load("/content/drive/MyDrive/best_model.pth", map_location=device)
model.load_state_dict(checkpoint)
model.eval()
print("Checkpoint loaded ✓")

In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

model.eval()
all_preds, all_labels = [], []
dice_per_class = {0: [], 1: [], 2: []}

with torch.no_grad():
    for seg_imgs, masks, seg_labels in tqdm(seg_test_loader):
        seg_imgs   = seg_imgs.to(device)
        masks      = masks.to(device)
        seg_labels = seg_labels.to(device)

        seg_pred, cls_pred = model(seg_imgs)

        # classification
        preds = cls_pred.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(seg_labels.cpu().numpy())

        # dice per class
        seg_true_sq = masks.squeeze(1).long()
        seg_prob    = torch.softmax(seg_pred, dim=1).argmax(dim=1)
        for c in range(3):
            pred_c = (seg_prob == c).float()
            true_c = (seg_true_sq == c).float()
            inter  = (pred_c * true_c).sum()
            dice   = (2 * inter) / (pred_c.sum() + true_c.sum() + 1e-8)
            dice_per_class[c].append(dice.item())

# ── Results ───────────────────────────────────────────────────────────
print("=== Segmentation Dice (per class) ===")
class_names_seg = ["Glioma", "Meningioma", "Pituitary"]
for c, name in enumerate(class_names_seg):
    print(f"  {name}: {np.mean(dice_per_class[c]):.4f}")
print(f"  Mean Dice: {np.mean([np.mean(v) for v in dice_per_class.values()]):.4f}")

print("\n=== Classification Report ===")
print(classification_report(all_labels, all_preds,
      target_names=["Glioma", "Meningioma", "Pituitary"]))

# ── Confusion Matrix ──────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Glioma", "Meningioma", "Pituitary"],
            yticklabels=["Glioma", "Meningioma", "Pituitary"])
plt.title("Confusion Matrix")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.show()

In [ ]:
# Check a few pituitary samples
pi_samples = [(img, mask, lbl) for img, mask, lbl in seg_test_dataset if lbl == 2]
print(f"Pituitary test samples: {len(pi_samples)}")
img, mask, lbl = pi_samples[0]
print(f"Mask unique values: {mask.unique()}")
print(f"Mask sum: {mask.sum()}")

In [ ]:
# Check mask values across different classes
for cls_code, idx in SEG_CLASS_MAP.items():
    samples = [(img, mask, lbl) for img, mask, lbl in seg_train_dataset if lbl == idx]
    mask = samples[0][1]
    print(f"{cls_code} mask unique values: {mask.unique()}, sum: {mask.sum():.0f}")

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

class MultiTaskUNet(nn.Module):
    def __init__(self, num_seg_classes=1, num_cls_classes=4):
        super().__init__()

        self.encoder = smp.encoders.get_encoder(
            "efficientnet-b3",
            in_channels=3,
            depth=5,
            weights="imagenet"
        )

        self.decoder = smp.decoders.unetplusplus.decoder.UnetPlusPlusDecoder(
            encoder_channels=self.encoder.out_channels,
            decoder_channels=(256, 128, 64, 32, 16),
            n_blocks=5,
            use_norm='batchnorm',
            center=False,
            attention_type="scse"
        )

        self.seg_head = smp.base.SegmentationHead(
            in_channels=16,
            out_channels=num_seg_classes,  # 1 = binary
            activation=None,
            kernel_size=3
        )

        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(self.encoder.out_channels[-1], 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_cls_classes)
        )

    def forward(self, x):
        features    = self.encoder(x)
        decoder_out = self.decoder(features)
        seg_mask    = self.seg_head(decoder_out)
        cls_out     = self.cls_head(features[-1])
        return seg_mask, cls_out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = MultiTaskUNet(num_seg_classes=1, num_cls_classes=4).to(device)
print("Model defined ✓")

In [ ]:
checkpoint = torch.load("/content/drive/MyDrive/best_model.pth", map_location=device)

# Remove old seg_head weights (shape mismatch: was 3, now 1)
filtered = {k: v for k, v in checkpoint.items() if not k.startswith("seg_head")}
missing, unexpected = model.load_state_dict(filtered, strict=False)
print(f"Missing keys:    {missing}")
print(f"Unexpected keys: {unexpected}")
print("Checkpoint loaded ✓")

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()
ce_cls   = nn.CrossEntropyLoss()

def combined_loss(seg_pred, seg_true, cls_pred, cls_true, alpha=0.5, beta=0.5):
    seg_l = bce_loss(seg_pred, seg_true)          # binary seg
    cls_l = ce_cls(cls_pred, cls_true)             # 4-class cls
    return alpha * seg_l + beta * cls_l

# Freeze encoder first
for param in model.encoder.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
print("Losses + optimizer ready ✓")

In [ ]:
from tqdm import tqdm

def train_epoch_finetune(model, seg_loader, cls_loader, optimizer):
    model.train()
    total_loss = 0
    cls_iter   = iter(cls_loader)

    for seg_imgs, masks, _ in tqdm(seg_loader, leave=False):
        seg_imgs = seg_imgs.to(device)
        masks    = masks.to(device)

        # get a cls batch (cycle if exhausted)
        try:
            cls_imgs, cls_labels = next(cls_iter)
        except StopIteration:
            cls_iter = iter(cls_loader)
            cls_imgs, cls_labels = next(cls_iter)

        cls_imgs   = cls_imgs.to(device)
        cls_labels = cls_labels.to(device)

        # seg forward
        seg_pred, _ = model(seg_imgs)
        # cls forward
        _, cls_pred = model(cls_imgs)

        loss = combined_loss(seg_pred, masks, cls_pred, cls_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(seg_loader)


def val_epoch_finetune(model, seg_loader, cls_loader):
    model.eval()
    total_loss  = 0
    correct     = 0
    total       = 0
    dice_scores = []
    cls_iter    = iter(cls_loader)

    with torch.no_grad():
        for seg_imgs, masks, _ in seg_loader:
            seg_imgs = seg_imgs.to(device)
            masks    = masks.to(device)

            try:
                cls_imgs, cls_labels = next(cls_iter)
            except StopIteration:
                cls_iter = iter(cls_loader)
                cls_imgs, cls_labels = next(cls_iter)

            cls_imgs   = cls_imgs.to(device)
            cls_labels = cls_labels.to(device)

            seg_pred, _  = model(seg_imgs)
            _, cls_pred  = model(cls_imgs)

            loss = combined_loss(seg_pred, masks, cls_pred, cls_labels)
            total_loss += loss.item()

            # cls accuracy
            preds    = cls_pred.argmax(dim=1)
            correct += (preds == cls_labels).sum().item()
            total   += cls_labels.size(0)

            # binary dice
            seg_prob = (torch.sigmoid(seg_pred) > 0.5).float()
            inter    = (seg_prob * masks).sum()
            dice     = (2 * inter) / (seg_prob.sum() + masks.sum() + 1e-8)
            dice_scores.append(dice.item())

    return total_loss / len(seg_loader), correct / total, np.mean(dice_scores)


# ── Fine-tune loop ─────────────────────────────────────────────────────
EPOCHS    = 10
best_dice = 0.0

# Unfreeze encoder after epoch 5
for epoch in range(EPOCHS):
    if epoch == 5:
        print("Unfreezing encoder...")
        for param in model.encoder.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

    train_loss = train_epoch_finetune(model, seg_train_loader, cls_train_loader, optimizer)
    val_loss, val_acc, val_dice = val_epoch_finetune(model, seg_test_loader, cls_test_loader)
    scheduler.step()

    print(f"Epoch {epoch+1:02d}/10 | "
          f"Train: {train_loss:.4f} | "
          f"Val: {val_loss:.4f} | "
          f"Acc: {val_acc:.4f} | "
          f"Dice: {val_dice:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "/content/drive/MyDrive/best_model_v2.pth")
        print(f"  → best model saved (dice: {best_dice:.4f})")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load best model
checkpoint = torch.load("/content/drive/MyDrive/best_model_v2.pth", map_location=device)
model.load_state_dict(checkpoint)
model.eval()

all_preds, all_labels = [], []
dice_scores = []

# ── Segmentation eval ─────────────────────────────────────────────────
with torch.no_grad():
    for seg_imgs, masks, _ in tqdm(seg_test_loader):
        seg_imgs = seg_imgs.to(device)
        masks    = masks.to(device)

        seg_pred, _ = model(seg_imgs)
        seg_prob    = (torch.sigmoid(seg_pred) > 0.5).float()
        inter       = (seg_prob * masks).sum()
        dice        = (2 * inter) / (seg_prob.sum() + masks.sum() + 1e-8)
        dice_scores.append(dice.item())

print(f"=== Segmentation ===")
print(f"  Mean Dice: {np.mean(dice_scores):.4f}")

# ── Classification eval ───────────────────────────────────────────────
with torch.no_grad():
    for cls_imgs, cls_labels in tqdm(cls_test_loader):
        cls_imgs   = cls_imgs.to(device)
        cls_labels = cls_labels.to(device)

        _, cls_pred = model(cls_imgs)
        preds       = cls_pred.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(cls_labels.cpu().numpy())

print(f"\n=== Classification Report ===")
print(classification_report(all_labels, all_preds,
      target_names=["Glioma", "Meningioma", "No Tumor", "Pituitary"]))

# ── Confusion Matrix ──────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Glioma", "Meningioma", "No Tumor", "Pituitary"],
            yticklabels=["Glioma", "Meningioma", "No Tumor", "Pituitary"])
plt.title("Confusion Matrix - Final Model")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()